# Evaluate code

In [28]:
from code_agent.creat_react_code_agent.code_agent_react import CodeAgentReact
from langsmith import evaluate, aevaluate

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


## Criando dataset no LangSmith

In [29]:
from datasets import load_dataset
from tqdm import tqdm

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)


Problems: 100%|██████████| 164/164 [00:00<00:00, 9150.56problem/s]


In [30]:
from langsmith import Client

client_langsmith = Client()


tamanho_amostra = 5

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(data_set_code_langsmith)), tamanho_amostra)
for index in index_aleatorios:
    data_aleatorios.append(data_set_code_langsmith[index])


# Create dataset if it doesn't exist
if not client_langsmith.has_dataset(dataset_name=dataset_name):
    dataset = client_langsmith.create_dataset(
        dataset_name=dataset_name, 
        description="The HumanEval dataset released by OpenAI includes 164 programming problems with a function sig- nature, docstring, body, and several unit tests. They were handwritten to ensure not to be included in the training set of code generation models."
    )
    
    client_langsmith.create_examples(dataset_id=dataset.id, examples=data_aleatorios)

In [31]:
data_aleatorios

[{'inputs': {'aswer_code': '\ndef cycpattern_check(a , b):\n    """You are given 2 words. You need to return True if the second word or any of its rotations is a substring in the first word\n    cycpattern_check("abcd","abd") => False\n    cycpattern_check("hello","ell") => True\n    cycpattern_check("whassup","psus") => False\n    cycpattern_check("abab","baa") => True\n    cycpattern_check("efef","eeff") => False\n    cycpattern_check("himenss","simen") => True\n\n    """\n'},
  'outputs': {'response_code': 'def check(candidate):\n\n    # Check some simple cases\n    #assert True, "This prints if this assert fails 1 (good for debugging!)"\n\n    # Check some edge cases that are easy to work out by hand.\n    #assert True, "This prints if this assert fails 2 (also good for debugging!)"\n    assert  candidate("xyzw","xyw") == False , "test #0"\n    assert  candidate("yello","ell") == True , "test #1"\n    assert  candidate("whattup","ptut") == False , "test #2"\n    assert  candidate("

In [32]:
dataset = client_langsmith.list_datasets()

### Avaliando o agente

In [33]:
from langsmith import traceable
from langchain_core.messages import HumanMessage

In [34]:
@traceable
async def agent_avaliado(question):
    answer = await agent.ainvoke(
    {
        "messages": 
            [HumanMessage(role="user",
                          content=question)],
        "todos": [],
    }
)
    return answer['messages'][-1].content


In [35]:
# We'll first define a custom code evaluator, which are useful to measure deterministic or close-ended metrics.
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200

LLM-as-a-Judge Evaluator
For open-ended metrics, it's can be powerful to use an LLM to score the outputs.

Let's use an LLM to check whether our application produces correct outputs. First, let's define a scoring schema for our LLM to adhere to in its response.

In [36]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score for the correctness of the answer, of an answer between 0 and 1")

In [37]:
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.huggingface import HuggingFaceModel
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

async def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

    <Rubric>
        A correct answer:
        - Provides accurate information
        - Uses suitable analogies and examples
        - Contains no factual errors
        - Is logically consistent

        When scoring, you should penalize:
        - Factual errors
        - Incoherent analogies and examples
        - Logical inconsistencies
    </Rubric>

    <Instructions>
        - Carefully read the input and output
        - Use the reference output to determine if the model output contains errors
        - Focus whether the model output uses accurate analogies and is logically consistent
    </Instructions>

    <Reminder>
        The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Use the reference outputs below to help you evaluate the correctness of the response:
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["aswer_code"], outputs["output"], reference_outputs["response_code"])

    
    """model = init_chat_model(model = "moonshotai/kimi-k2-instruct-0905", model_provider = "nvidia")
    
    model_stutured = model.with_structured_output(CorrectnessScore)
    
    response = await model_stutured.ainvoke([HumanMessage(content=prompt)])"""
    
    model = GoogleModel('models/gemini-2.0-flash-lite')
    agent = Agent(model, output_type=CorrectnessScore)

    try:
        response = agent.run_sync(prompt)
    except Exception as e:
        logger.error(f"Erro do tipo: {e}")
        model = GoogleModel('models/gemini-2.0-flash')
        agent = Agent(model, output_type=CorrectnessScore)
        response = agent.run_sync(prompt)
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
    return response

In [38]:
import nest_asyncio

# Apply nest_asyncio at the start of your notebook
nest_asyncio.apply()

def correctness_sync(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    return asyncio.run(correctness(inputs, outputs, reference_outputs))

In [39]:
response = correctness_sync({"aswer_code": "def add(a, b):\n    return a + b"}, {"output": "def add(a, b):\n    return a + b"}, {"response_code": "def add(a, b):\n    return a + b"})

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"


In [40]:
response

1

In [41]:
# 4. Define a function to run your application
async def run(inputs: dict):
    return await agent_avaliado(inputs["aswer_code"])


In [ ]:
from langsmith import evaluate, aevaluate

modelos = [
    #{"model": "openai/gpt-oss-20b",  "provider": "groq"},
    #{"model": "meta-llama/llama-4-scout-17b-16e-instruct",  "provider": "groq"},
    #{"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "groq"},
    #{"model": "qwen/qwen3-next-80b-a3b-instruct",  "provider": "nvidia"},
    #{"model": "deepseek-ai/deepseek-v3.1",  "provider": "nvidia"},
    #{"model": "microsoft/phi-4-mini-instruct",  "provider": "nvidia"},
    #{"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "nvidia"},
    #{"model": "meta/llama-4-scout-17b-16e-instruct",  "provider": "nvidia"},
    #{"model": "openai/gpt-oss-120b",  "provider": "nvidia"},
    #{"model": "moonshotai/kimi-k2-instruct",  "provider": "nvidia"},
    #{"model": "meta/llama-3.3-70b-instruct",  "provider": "nvidia"},
    #{"model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",  "provider": "nvidia"},
    #{"model": "nvidia/llama-3.1-nemotron-ultra-253b-v1",  "provider": "nvidia"},
    #{"model": "meta/llama-3.1-405b-instruct",  "provider": "nvidia"},
    #{"model": "nvidia/llama-3.1-nemotron-nano-4b-v1.1",  "provider": "nvidia"},
    #{"model": "ibm/granite-3.3-8b-instruct",  "provider": "nvidia"},
    #{"model": "nv-mistralai/mistral-nemo-12b-instruct",  "provider": "nvidia"},
    #{"model": "nvidia/llama-3.3-nemotron-super-49b-v1",  "provider": "nvidia"},
    #{"model": "mistralai/mistral-small-3.1-24b-instruct-2503",  "provider": "nvidia"},
    #{"model": "qwen/qwq-32b",  "provider": "nvidia"},
    #{"model": "meta/llama-3.1-8b-instruct",  "provider": "nvidia"},
    #{"model": "mistralai/mistral-nemotron",  "provider": "nvidia"},
    #{"model": "meta/llama-3.2-3b-instruct",  "provider": "nvidia"},
    {"model": "openai/gpt-oss-20b",  "provider": "nvidia"},
    {"model": "mistralai/mistral-large-2-instruct",  "provider": "nvidia"},
    {"model": "deepseek-ai/deepseek-r1-0528",  "provider": "nvidia"},
    {"model": "meta/llama-3.1-70b-instruct",  "provider": "nvidia"}
    
    
]

models_concluidos = []

for modelo in modelos:
    print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provider']}")
    
    if modelo['model'] in models_concluidos:
        print("Modelo já concluído")
        continue
    else:
        code_agent = CodeAgentReact(model=modelo['model'], model_provider=modelo['provider'])
        agent = code_agent.create_agent()
        
        @traceable
        async def agent_avaliado(question):
            answer = await agent.ainvoke(
            {
                "messages": 
                    [HumanMessage(role="user",
                                content=question)],
                "todos": [],
            }
        )
            return answer['messages'][-1].content
        
        async def run(inputs: dict):
            return await agent_avaliado(inputs["aswer_code"])
        
        
        results = asyncio.run(aevaluate(
            run,
            data=dataset_name,
            evaluators=[correctness, conciseness],
            experiment_prefix=f"{dataset_name}-{modelo['model']}-{modelo['provider']}"))
        
        models_concluidos.append(modelo['model']) 

### Avaliando os resultados

In [42]:
from langsmith import Client
client_langsmith = Client()

In [43]:
experiments_names = [experiment.name for experiment in client_langsmith.list_projects() if "Human-Eval-Code-" in experiment.name]
experiments_names

['Human-Eval-Code-5-aleatorios-moonshotai/kimi-k2-instruct-0905-nvidia-2d202f43',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7e8431d8',
 'Human-Eval-Code-5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-17f68b12',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-large-2-instruct-nvidia-12c3ca39',
 'Human-Eval-Code-5-aleatorios-openai/gpt-oss-20b-nvidia-df66f777',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-3d339d56',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-nemotron-nvidia-44484a26',
 'Human-Eval-Code-5-aleatorios-meta/llama-3.1-8b-instruct-nvidia-5ec1633b',
 'Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvidia-c5ffcc47',
 'Human-Eval-Code-5-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-6a6f639d',
 'Human-Eval-Code-5-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1-nvidia-7ec57602',
 'Human-Eval-Code-5-aleatorios-nv-mistralai/mistral-nemo-12b-instruct-nvidia-68e1ffbc',
 'Human-Eval-Code-5-aleatorios-ibm/granite-3.3-8b-

In [44]:
# Set this to load expt results
for experiment_name in experiments_names:
    print("Loading results for experiment:", experiment_name)
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", experiment_results.latency_p50)
    print("Latency p99:", experiment_results.latency_p99)
    print("Token Usage:", experiment_results.total_tokens)
    print("Feedback Stats:", experiment_results.feedback_stats)
    print("*" * 50)

Loading results for experiment: Human-Eval-Code-5-aleatorios-moonshotai/kimi-k2-instruct-0905-nvidia-2d202f43
Latency p50: 0:04:43.158000
Latency p99: 0:04:43.158000
Token Usage: 76945
Feedback Stats: None
**************************************************
Loading results for experiment: Human-Eval-Code-5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7e8431d8
Latency p50: 0:00:29.496000
Latency p99: 0:01:21.206000
Token Usage: 45524
Feedback Stats: {'conciseness': {'n': 5, 'avg': 0.8, 'stdev': 0.39999999999999997, 'errors': 0, 'values': {}, 'type': 'primary'}, 'correctness': {'n': 5, 'avg': 0.4, 'stdev': 0.4898979485566356, 'errors': 0, 'values': {}, 'type': 'primary'}}
**************************************************
Loading results for experiment: Human-Eval-Code-5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-17f68b12
Latency p50: 0:00:38.971000
Latency p99: 0:02:08.436400
Token Usage: 26395
Feedback Stats: {'conciseness': {'n': 5, 'avg': 0.8, 'stdev': 0.39999999999999997, 'error

In [45]:
import pandas as pd
data_frames = pd.DataFrame()
for experiment_name in experiments_names:
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    name_column = experiment_name.split("Human-Eval-Code-")[1]
    data = pd.DataFrame.from_dict(experiment_results.dict()).T
    try:
        data_frames[name_column] = data["correctness"]
    except Exception as e:
        print(e)
        pass
    

'correctness'
'correctness'
'correctness'


In [46]:
data_frames

,5-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7e8431d8,5-aleatorios-deepseek-ai/deepseek-r1-0528-nvidia-17f68b12,5-aleatorios-mistralai/mistral-large-2-instruct-nvidia-12c3ca39,5-aleatorios-openai/gpt-oss-20b-nvidia-df66f777,5-aleatorios-meta/llama-3.2-3b-instruct-nvidia-3d339d56,5-aleatorios-mistralai/mistral-nemotron-nvidia-44484a26,5-aleatorios-qwen/qwq-32b-nvidia-c5ffcc47,5-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-6a6f639d,5-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1-nvidia-7ec57602,5-aleatorios-nv-mistralai/mistral-nemo-12b-instruct-nvidia-68e1ffbc,...,5-aleatorios-moonshotai/kimi-k2-instruct-nvidia-898fd267,5-aleatorios-openai/gpt-oss-120b-nvidia-972586a4,5-aleatorios-meta/llama-4-scout-17b-16e-instruct-nvidia-eb0f3f28,5-aleatorios-moonshotai/kimi-k2-instruct-0905-nvidia-94cdcbd1,5-aleatorios-microsoft/phi-4-mini-instruct-nvidia-103a58cc,5-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-c3468f9f,5-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-b6c6da97,5-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-759669dd,5-aleatorios-meta-llama/llama-4-scout-17b-16e-instruct-groq-15a85c28,5-aleatorios-openai/gpt-oss-20b-groq-87adb391
id,6602d14d-6524-4b44-b436-a2e23b2a8d61,00624a2f-c81a-437d-b29e-46e10b710fe7,99490f67-7191-420e-9cf7-5cb7bb9f38dc,992c20fc-de59-4da1-a92c-11a63a39049a,708b59ee-19af-4c9c-908b-3876f9e651a2,cf1fc542-b503-462e-b7c0-3532175fd4d4,78793346-be81-4829-bdde-8ebd0ab83e28,54079331-7bc8-4a11-8f80-3edae5867b85,fc51cfa7-6465-4e40-8a2e-9581a0162beb,9f722ea2-2480-4c84-97a9-eed451b61a3f,...,92d1d243-ac05-41cf-aa91-651d12e2d20e,d128a1f4-a634-4586-b842-b4fd246c6708,3bf23156-2f77-40ba-82cb-6c2476ef95df,26d48eab-ead5-412a-909c-b66ea3bde1bf,fd5fe5b9-6915-401d-b758-2f3802325580,a3acd42f-e6d8-45b8-90e3-4bbbcd2d68e8,536ee10a-1c28-4d0a-8045-a2ce1f78ea91,e572a9e8-48a6-4fb1-b1cf-3c77e4e0de18,e01c1a84-5d46-4dc1-8f4a-db2b2943821c,3501c9f2-db0a-4f4b-a541-add25988d865
start_time,2025-10-14 16:47:40.476408+00:00,2025-10-14 16:42:42.843531+00:00,2025-10-14 16:42:23.558368+00:00,2025-10-14 16:41:44.626107+00:00,2025-10-14 15:17:42.387629+00:00,2025-10-14 15:16:51.839518+00:00,2025-10-14 14:06:38.415047+00:00,2025-10-14 14:04:32.540711+00:00,2025-10-14 13:53:35.913451+00:00,2025-10-14 13:29:18.356226+00:00,...,2025-10-14 00:50:10.998266+00:00,2025-10-14 00:46:17.308383+00:00,2025-10-14 00:40:46.879063+00:00,2025-10-14 00:19:50.561951+00:00,2025-10-13 17:48:03.618602+00:00,2025-10-13 17:21:32.823472+00:00,2025-10-13 17:03:42.605934+00:00,2025-10-13 16:52:59.349572+00:00,2025-10-13 16:50:00.062808+00:00,2025-10-13 16:43:08.521425+00:00
end_time,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
description,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
name,Human-Eval-Code-5-aleatorios-meta/llama-3.1-70...,Human-Eval-Code-5-aleatorios-deepseek-ai/deeps...,Human-Eval-Code-5-aleatorios-mistralai/mistral...,Human-Eval-Code-5-aleatorios-openai/gpt-oss-20...,Human-Eval-Code-5-aleatorios-meta/llama-3.2-3b...,Human-Eval-Code-5-aleatorios-mistralai/mistral...,Human-Eval-Code-5-aleatorios-qwen/qwq-32b-nvid...,Human-Eval-Code-5-aleatorios-mistralai/mistral...,Human-Eval-Code-5-aleatorios-nvidia/llama-3.3-...,Human-Eval-Code-5-aleatorios-nv-mistralai/mist...,...,Human-Eval-Code-5-aleatorios-moonshotai/kimi-k...,Human-Eval-Code-5-aleatorios-openai/gpt-oss-12...,Human-Eval-Code-5-aleatorios-meta/llama-4-scou...,Human-Eval-Code-5-aleatorios-moonshotai/kimi-k...,Human-Eval-Code-5-aleatorios-microsoft/phi-4-m...,Human-Eval-Code-5-aleatorios-deepseek-ai/deeps...,Human-Eval-Code-5-aleatorios-qwen/qwen3-next-8...,Human-Eval-Code-5-aleatorios-moonshotai/kimi-k...,Human-Eval-Code-5-aleatorios-meta-llama/llama-...,Human-Eval-Code-5-aleatorios-openai/gpt-oss-20...
extra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenant_id,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961

#### Selecionando os melhores modelos e Testando com mais exemplos

In [47]:
melhores_experiments = [colunas for colunas in data_frames.columns if data_frames[colunas]['feedback_stats']['avg'] > 0.6]
data_frames_melhores = data_frames[melhores_experiments]
data_frames_melhores.head()

,5-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-6a6f639d,5-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-340e3750,5-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1.5-nvidia-eb893acb,5-aleatorios-meta/llama-4-scout-17b-16e-instruct-nvidia-eb0f3f28,5-aleatorios-moonshotai/kimi-k2-instruct-0905-nvidia-94cdcbd1,5-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-c3468f9f,5-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-b6c6da97
id,54079331-7bc8-4a11-8f80-3edae5867b85,9db424b0-ae0a-4ebf-8328-7ec253f76d1b,11c05d0f-b3c9-4de0-b024-1de60dbb7f1c,3bf23156-2f77-40ba-82cb-6c2476ef95df,26d48eab-ead5-412a-909c-b66ea3bde1bf,a3acd42f-e6d8-45b8-90e3-4bbbcd2d68e8,536ee10a-1c28-4d0a-8045-a2ce1f78ea91
start_time,2025-10-14 14:04:32.540711+00:00,2025-10-14 01:01:30.956791+00:00,2025-10-14 00:57:41.016777+00:00,2025-10-14 00:40:46.879063+00:00,2025-10-14 00:19:50.561951+00:00,2025-10-13 17:21:32.823472+00:00,2025-10-13 17:03:42.605934+00:00
end_time,None,None,None,None,None,None,None
description,None,None,None,None,None,None,None
name,Human-Eval-Code-5-aleatorios-mistralai/mistral...,Human-Eval-Code-5-aleatorios-nvidia/llama-3.1-...,Human-Eval-Code-5-aleatorios-nvidia/llama-3.3-...,Human-Eval-Code-5-aleatorios-meta/llama-4-scou...,Human-Eval-Code-5-aleatorios-moonshotai/kimi-k...,Human-Eval-Code-5-aleatorios-deepseek-ai/deeps...,Human-Eval-Code-5-aleatorios-qwen/qwen3-next-8...


In [48]:
melhores_modelos = [{"model":coluna.split('-aleatorios-')[1].rsplit('-', 2)[0], "provedor":coluna.split('-')[-2]} for coluna in data_frames_melhores.columns]

In [49]:
from datasets import load_dataset
from tqdm import tqdm

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)

Problems: 100%|██████████| 164/164 [00:00<00:00, 14256.29problem/s]


In [50]:
from langsmith import Client

client_langsmith = Client()


tamanho_amostra = 15

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(data_set_code_langsmith)), tamanho_amostra)
for index in index_aleatorios:
    data_aleatorios.append(data_set_code_langsmith[index])


# Create dataset if it doesn't exist
if not client_langsmith.has_dataset(dataset_name=dataset_name):
    dataset = client_langsmith.create_dataset(
        dataset_name=dataset_name, 
        description="The HumanEval dataset released by OpenAI includes 164 programming problems with a function sig- nature, docstring, body, and several unit tests. They were handwritten to ensure not to be included in the training set of code generation models."
    )
    
    client_langsmith.create_examples(dataset_id=dataset.id, examples=data_aleatorios)

In [51]:
experiments_realizados = [experiment.name.split('-aleatorios-')[1].rsplit('-', 2)[0] for experiment in client_langsmith.list_projects() if "Human-Eval-Code-15" in experiment.name]

experiments_realizados

[]

In [ ]:
models_concluidos_melhores = []

for modelo in melhores_modelos:
    print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provedor']}")
    
    if modelo['model'] in experiments_realizados:
        print("Modelo já concluído")
        continue
    else:
        code_agent = CodeAgentReact(model=modelo['model'], model_provider=modelo['provedor'])
        agent = code_agent.create_agent()
        
        @traceable
        async def agent_avaliado(question):
            answer = await agent.ainvoke(
            {
                "messages": 
                    [HumanMessage(role="user",
                                content=question)],
                "todos": [],
            }
        )
            return answer['messages'][-1].content
        
        async def run(inputs: dict):
            return await agent_avaliado(inputs["aswer_code"])
        
        
        results = asyncio.run(aevaluate(
            run,
            data=dataset_name,
            evaluators=[correctness, conciseness],
            experiment_prefix=f"{dataset_name}-{modelo['model']}-{modelo['provedor']}"))
        
        models_concluidos_melhores.append(modelo['model']) 

INFO:code_agent.creat_react_code_agent.code_agent_react:CodeAgentReact inicializado: provider=nvidia, checkpointer=False
INFO:code_agent.creat_react_code_agent.code_agent_react:Modelo provider=nvidia inicializado com sucesso


Rodando o agente com o modelo mistralai/mistral-small-3.1-24b-instruct-2503 do provedor nvidia


INFO:code_agent.creat_react_code_agent.code_agent_react:Agente React criado com sucesso


View the evaluation results for experiment: 'Human-Eval-Code-15-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-5c8c6900' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/2129ac60-d387-4965-a732-fdc5e4e27759/compare?selectedSessions=daf0c32e-4e3b-498d-b747-89d81f4d417d




0it [00:00, ?it/s]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
1it [00:26, 26.88s/it]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
2it [00:42, 20.44s/it]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
3it [01:19, 27.64s/it]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
4it [01:43, 26.54s/it]INFO:google_genai.models:AFC is enabled with max remote ca

...

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
2it [01:34, 52.62s/it]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
3it [01:50, 36.08s/it]INFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:code_agent.get_routem_llm.routem_llm:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507
INFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET

🤔 AGENTE PENSANDO...
--------------------------------------------------
The user wants a function to calculate maximum nesting depth of parentheses in space-separated groups. From the inputs, I can see there's already a code implementation that's been reviewed. The supervisor feedback indicates the code needs changes - specifically adding validation to ensure only parentheses characters are present in each group, and potentially making error behavior configurable.

Let me analyze what needs to be implemented:
1. Main function that takes a space-separated string
2. Split into groups by spaces
3. For each group, calculate max nesting depth of parentheses
4. Handle edge cases: empty input, unbalanced parentheses, non-parentheses characters
5. Return list of max depths for each group

The current code has good structure but needs the validation enhancement mentioned in the feedback. I should provide the improved version that includes validation for invalid characters and explain the soluti

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:code_agent.get_routem_llm.routem_llm:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507
INFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
ERROR:code_agent.get_routem_llm.router_cerebras:Erro no llm_pydanticai: Error code: 429 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
INFO:httpx:HTTP Request: GET https://api.cerebr

🤔 AGENTE PENSANDO...
--------------------------------------------------
The user wants me to implement a function that parses nested parentheses and returns the maximum nesting depth. I need to:

1. Create a function called parse_nested_parens that takes a string input
2. Add proper input validation (type checking, character validation, balance checking)
3. Handle invalid characters appropriately
4. Include test cases as shown in the docstring examples
5. Make sure the function raises appropriate exceptions with clear error messages

Looking at the provided code, it already seems to be a complete implementation with all the required features. The code includes:
- Proper docstring with examples
- Type validation (raises TypeError for non-string input)
- Character validation (raises ValueError for invalid characters)
- Balance checking (raises ValueError for unbalanced parentheses)
- Correct algorithm to calculate maximum nesting depth

The implementation looks correct and complete. I sh

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


### Acessando datasets do LangSmith

In [ ]:
# List all datasets
from langsmith import Client

client = Client()

datasets = client.list_datasets()
for dataset in datasets:
    print(dataset.id, dataset.name)

2129ac60-d387-4965-a732-fdc5e4e27759 Human-Eval-Code-15-aleatorios
4dfd7fc2-d782-4c4d-ba17-87f5a251f9a7 Human-Eval-Code-5-aleatorios
b9dd1aaa-017b-4280-96ca-cd6f094773fa deep_research_supervisor_parallelism
5873a0fe-22fd-4a3d-8eb6-86330c26e52f deep_research_agent_termination
2bdcce58-182b-4a36-98a5-74720bc26a05 deep_research_scoping
27916b23-40e0-4874-8d5c-ddbebfcb8958 E-mail Triage Evaluation
b746e978-0744-44de-9f42-6cbd42bbd1e3 agents-from-scratch.test_response
623cff70-db0b-4993-bfe4-6ee88d1a46ac agents-from-scratch.test_tools
a2defcc0-e281-43c6-b351-c7930a9307ac Financial Advisory RAG Evaluation
77fa7a2a-4ece-4096-96fa-ed9d1d9cff81 Healthcare Agent Trajectory Evaluation
43be1111-6a51-4928-a33e-940a33a8b40a Reasoning and Bias
ba1609ff-70a6-4a64-b849-1892bbd24274 QA Example Dataset
aea62e89-af3d-4bb0-98e6-275fd643bfe5 Sample dataset
da848d07-6fa9-4ffe-a503-570e10840ac0 Insurance Claims


In [ ]:
examples = client.list_examples(dataset_id="0b278319-9299-4f5c-8ac3-d068c769469f")
for example in examples:
    print(example.inputs, example.outputs)

LangSmithNotFoundError: Resource not found for /examples. HTTPError('404 Client Error: Not Found for url: https://api.smith.langchain.com/examples?offset=0&inline_s3_urls=True&limit=100&dataset=0b278319-9299-4f5c-8ac3-d068c769469f', '{"detail":"dataset 0b278319-9299-4f5c-8ac3-d068c769469f not found"}')

In [ ]:
experiment_results.dict()

{'id': UUID('e9d43d28-170e-4122-9546-956cf5dec521'),
 'start_time': datetime.datetime(2025, 10, 3, 15, 18, 51, 697051, tzinfo=datetime.timezone.utc),
 'end_time': None,
 'description': None,
 'name': 'test-human-eval-5-qwen/qwen3-next-80b-a3b-instruct-nvidia-dc157e0e',
 'extra': {'metadata': {'git': {'tags': None,
    'dirty': True,
    'branch': 'secundario',
    'commit': '47b18b6248c690a39e12ddf4182201433d49be46',
    'repo_name': 'AgenteCodificaoLangGraph',
    'remote_url': 'https://github.com/Jeferson100/Code-Agent.git',
    'author_name': 'JEFERSON DIONEI SEHNEM',
    'commit_time': '1759163044',
    'author_email': 'sehnemjeferson@gmail.com'},
   'revision_id': '47b18b6-dirty',
   'dataset_splits': ['base'],
   'dataset_version': '2025-10-02T13:48:42.613843+00:00',
   'num_repetitions': 1}},
 'tenant_id': UUID('d9ab5018-45a8-5efd-961d-320c87839c87'),
 'reference_dataset_id': UUID('7556c14d-cd6e-4aaf-9bec-979228fa1f71'),
 'run_count': 5,
 'latency_p50': datetime.timedelta(second